## Transform Results Data
- Read bronze Results table
- Keep only the columns required for analytics(Drop url column)
- Standardise column name using snakecase(constructorId->constructor_id,driverId->driver_id)
- Rename columns to make them more meaningful(grid->grid_position)
- Filter out rows where seaon,round,constructor_id and driver_id is null (business key validation)
- Remove duplicate records.
- Transform values of column race_name to Title Case
- Write the transformed data to silver results table

In [0]:
dbutils.widgets.text("p_batch_id","")
v_batch_id=dbutils.widgets.get("p_batch_id")

In [0]:
%run ../00-common/03.silver-help

In [0]:
%run ../00-common/01.environment-config

In [0]:
bronze_table=f"{catalog_name}.{bronze_schema}.results"
silver_table=f"{catalog_name}.{silver_schema}.results"

In [0]:
import pyspark.sql.functions as F

### Step 1,2,3&4 Read bronze results table,select only required colum and standardise column names

In [0]:
results_df=(
    spark.table(bronze_table)
    .drop("url")
    .withColumnsRenamed(
        {"constructorId":"constructor_id",
         "driverId":"driver_id",
         "raceName":"race_name",
         "date":"race_date",
         "time":"race_time",
         "grid":"grid_position",
         "laps":"completed_laps",
         "number":"car_number",
         "position":"final_position",
         "positionText":"final_position_text"
        }
    )
)
    


### Step 5 & 6  Apply Data Quality Checks
- Filter out rows where seaon,round,constructor_id and driver_id is null (business key validation)
- Remove Duplicates

In [0]:
results_valid_df=(
    results_df.filter(
        F.col("season").isNotNull() &
        F.col("round").isNotNull() &
        F.col("driver_id").isNotNull() &
        F.col("constructor_id").isNotNull()
    )
    .dropDuplicates(["season","round","driver_id","constructor_id"])
)



In [0]:

display(results_valid_df)



### Step -7 Transform values of column race_name to Title Case

In [0]:
results_final_df=(
    results_valid_df
    .withColumn("race_name",F.initcap(F.col("race_name")))
)
#display(drivers_final_df)

### Step-8 Write the transformed data to silver drivers table

In [0]:
write_to_silver(
    results_final_df,
    silver_table,
    merge_condition="s.season=t.season AND s.round=t.round AND s.constructor_id=t.constructor_id AND s.driver_id =t.driver_id",
    columns_to_update=[
       "race_date",
       "race_name",
        "round",
        "season",
       "constructor_id",
       "driver_id",
       "grid_position",
      "completed_laps",
       "car_number",
       "points",
       "final_position",
       "final_position_text",
       "status",
       "ingestion_timestamp",
        "source_file",
        "batch_id" 
    ]
)

In [0]:
display(spark.read.table(silver_table))

race_date,race_name,round,season,constructor_id,driver_id,grid_position,completed_laps,car_number,points,final_position,final_position_text,status,ingestion_timestamp,source_file,batch_id,created_timestamp,updated_timestamp
2024-03-02,Bahrain Grand Prix,1,2024,red_bull,perez,5,57,11,18.0,2,2,Finished,2026-10-20T07:42:15.718Z,dbfs:/Volumes/formula1_incr/landing/files/2025-01/results/results_2024.json,2025-01,2026-10-20T11:41:22.145Z,2026-10-20T11:41:55.591Z
2024-03-02,Bahrain Grand Prix,1,2024,mercedes,hamilton,9,57,44,6.0,7,7,Finished,2026-10-20T07:42:15.718Z,dbfs:/Volumes/formula1_incr/landing/files/2025-01/results/results_2024.json,2025-01,2026-10-20T11:41:22.145Z,2026-10-20T11:41:55.591Z
2024-03-02,Bahrain Grand Prix,1,2024,mclaren,piastri,8,57,81,4.0,8,8,Finished,2026-10-20T07:42:15.718Z,dbfs:/Volumes/formula1_incr/landing/files/2025-01/results/results_2024.json,2025-01,2026-10-20T11:41:22.145Z,2026-10-20T11:41:55.591Z
2024-03-02,Bahrain Grand Prix,1,2024,rb,ricciardo,14,56,3,0.0,13,13,Lapped,2026-10-20T07:42:15.718Z,dbfs:/Volumes/formula1_incr/landing/files/2025-01/results/results_2024.json,2025-01,2026-10-20T11:41:22.145Z,2026-10-20T11:41:55.591Z
2024-04-07,Japanese Grand Prix,4,2024,ferrari,sainz,4,53,55,15.0,3,3,Finished,2026-10-20T07:42:15.718Z,dbfs:/Volumes/formula1_incr/landing/files/2025-01/results/results_2024.json,2025-01,2026-10-20T11:41:22.145Z,2026-10-20T11:41:55.591Z
2024-04-07,Japanese Grand Prix,4,2024,mclaren,piastri,6,53,81,4.0,8,8,Finished,2026-10-20T07:42:15.718Z,dbfs:/Volumes/formula1_incr/landing/files/2025-01/results/results_2024.json,2025-01,2026-10-20T11:41:22.145Z,2026-10-20T11:41:55.591Z
2024-04-21,Chinese Grand Prix,5,2024,aston_martin,alonso,3,56,14,7.0,7,7,Finished,2026-10-20T07:42:15.718Z,dbfs:/Volumes/formula1_incr/landing/files/2025-01/results/results_2024.json,2025-01,2026-10-20T11:41:22.145Z,2026-10-20T11:41:55.591Z
2024-05-05,Miami Grand Prix,6,2024,ferrari,leclerc,2,57,16,15.0,3,3,Finished,2026-10-20T07:42:15.718Z,dbfs:/Volumes/formula1_incr/landing/files/2025-01/results/results_2024.json,2025-01,2026-10-20T11:41:22.145Z,2026-10-20T11:41:55.591Z
2024-05-05,Miami Grand Prix,6,2024,alpine,ocon,13,57,31,1.0,10,10,Finished,2026-10-20T07:42:15.718Z,dbfs:/Volumes/formula1_incr/landing/files/2025-01/results/results_2024.json,2025-01,2026-10-20T11:41:22.145Z,2026-10-20T11:41:55.591Z
2024-05-26,Monaco Grand Prix,8,2024,rb,tsunoda,8,77,22,4.0,8,8,Lapped,2026-10-20T07:42:15.718Z,dbfs:/Volumes/formula1_incr/landing/files/2025-01/results/results_2024.json,2025-01,2026-10-20T11:41:22.145Z,2026-10-20T11:41:55.591Z
